# Practice I: Interactive Personal Budget Manager

**Teun Schulten**  
Master in Technology and Business — Programming with Python  
Release 1: a reliable core

This notebook records **expected behaviour**, then runs the **tests** and **CLI flows** so the results stay visible as evidence. Nothing here is submitted automatically; it is a walk-through of the project in `src/budget_manager`.


## 1. Setup

The first code cell moves into the project folder and puts `src` on `PYTHONPATH`, so `python -m budget_manager` and `import budget_manager` both work.


In [1]:
import os
import sys
import subprocess
import tempfile
from pathlib import Path

PROJECT = Path.cwd().resolve()
if not (PROJECT / "src" / "budget_manager").is_dir():
    for candidate in [PROJECT, *PROJECT.parents]:
        if (candidate / "src" / "budget_manager").is_dir():
            PROJECT = candidate
            break
    else:
        raise FileNotFoundError(
            "Could not find the project folder (expected src/budget_manager). "
            "Open the notebook from Practice_I_Teun_Schulten/."
        )

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT / "src")

print("Project folder:", "Practice_I_Teun_Schulten")
print("Python:", sys.executable)
print("Package import OK:" , end=" ")
import budget_manager
print(budget_manager.__version__)


Project folder: Practice_I_Teun_Schulten
Python: /workspace/aiml/.venv/bin/python3
Package import OK: 1.0.0


## 2. Expected behaviour (Think → Predict → Test)

The brief asks to define expected behaviour before running code. These are the predictions the tests and CLI cells below should match.

### Normal case

| Action | Expected result |
|--------|-----------------|
| Add expense `food` 18.50 EUR on 2026-09-04, then income `salary` 2500 EUR on 2026-09-05 | Ids `T001` and `T002` are assigned. |
| `list` | Two aligned lines, all six fields present. |
| `report` | 2 transactions, income 2500.00, expense 18.50, balance **2481.50** EUR. |

### Boundary cases

| Input | Expected message / behaviour |
|-------|------------------------------|
| `amount=-5` | `amount must be > 0.` — the list stays empty. |
| `category=snacks` | `category must be one of: education, entertainment, food, freelance, gift, health, rent, salary, shopping, transport, utilities.` |
| `currency=eur` | `currency must be a 3-letter ISO code (e.g. EUR).` — not silently changed to `EUR`. |
| `amount=0` | `amount must be > 0.` |
| `date=2026-13-01` | `date must be a real calendar date in ISO format YYYY-MM-DD.` |
| Import `data/sample_transactions.csv` | **5** rows imported (`T001`, `T002`, `T009`, `T010`, `T011`). **Row 4:** `amount must be > 0.` plus five further row errors. EUR balance **1829.10**. Exit code 1 because some rows failed. |
| Undo after save, crash, and reload | Last saved id is removed from the file on disk, not only from memory. |
| Empty undo | `Nothing to undo.` |
| 100 EUR + 50 USD | Totals stay in separate currency groups; they are never added together. |

Category policy: unknown values are **rejected**, not mapped to a generic bucket.


## 3. Project layout

One module, one responsibility. The CLI contains no business rules.


In [2]:
skip = {"__pycache__", ".venv", ".git", ".ipynb_checkpoints", "final"}

def show_tree(path: Path, prefix=""):
    entries = sorted(
        [p for p in path.iterdir() if p.name not in skip and not p.name.endswith(".pyc")],
        key=lambda p: (p.is_file(), p.name.lower()),
    )
    for i, entry in enumerate(entries):
        branch = "└── " if i == len(entries) - 1 else "├── "
        print(f"{prefix}{branch}{entry.name}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            show_tree(entry, prefix + extension)

print("Practice_I_Teun_Schulten/")
show_tree(PROJECT)


Practice_I_Teun_Schulten/
├── data
│   └── sample_transactions.csv
├── src
│   └── budget_manager
│       ├── __init__.py
│       ├── __main__.py
│       ├── cli.py
│       ├── models.py
│       ├── services.py
│       ├── storage.py
│       └── validation.py
├── tests
│   └── test_budget_manager.py
├── .gitignore
├── AI_DISCLOSURE.md
├── DESIGN.md
├── Practice_I_Teun_Schulten.html
├── Practice_I_Teun_Schulten.ipynb
└── README.md


## 4. Tests

`unittest` only — no extra packages. The file `tests/test_budget_manager.py` covers the normal case and the boundaries in the table above.


In [3]:
result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=PROJECT,
    capture_output=True,
    text=True,
)
# unittest writes the runner output to stderr
print(result.stderr or result.stdout)
print("exit code:", result.returncode)
assert result.returncode == 0, "Tests failed"


test_csv_import_reports_row_errors_and_keeps_valid_rows (test_budget_manager.BoundaryTests.test_csv_import_reports_row_errors_and_keeps_valid_rows) ... ok
test_invalid_date_is_rejected (test_budget_manager.BoundaryTests.test_invalid_date_is_rejected) ... ok
test_lowercase_currency_is_not_silently_fixed (test_budget_manager.BoundaryTests.test_lowercase_currency_is_not_silently_fixed) ... ok
test_mixed_currencies_are_not_added_together (test_budget_manager.BoundaryTests.test_mixed_currencies_are_not_added_together) ... ok
test_negative_amount_is_rejected (test_budget_manager.BoundaryTests.test_negative_amount_is_rejected) ... ok
test_unknown_category_is_rejected (test_budget_manager.BoundaryTests.test_unknown_category_is_rejected) ... ok
test_zero_amount_is_rejected (test_budget_manager.BoundaryTests.test_zero_amount_is_rejected) ... ok
test_interactive_add_rejects_negative_and_saves_one (test_budget_manager.InteractiveTests.test_interactive_add_rejects_negative_and_saves_one) ... ok
tes

## 5. CLI evidence

The next cells run the real command-line interface in a temporary folder, so the project directory is not modified. `--file` must come **before** the subcommand.


In [4]:
def run_cli(*args, workdir, filename="transactions.json"):
    command = [sys.executable, "-m", "budget_manager", "--file", filename, *args]
    completed = subprocess.run(
        command,
        cwd=workdir,
        capture_output=True,
        text=True,
        env={**os.environ, "PYTHONPATH": str(PROJECT / "src")},
    )
    shown = []
    for arg in args:
        try:
            shown.append(str(Path(arg).resolve().relative_to(PROJECT)))
        except (ValueError, OSError):
            shown.append(arg)
    display = " ".join(["python", "-m", "budget_manager", "--file", filename, *shown])
    print("$", display)
    if completed.stdout:
        print(completed.stdout, end="" if completed.stdout.endswith("\n") else "\n")
    if completed.stderr:
        print(completed.stderr, end="" if completed.stderr.endswith("\n") else "\n")
    print(f"(exit {completed.returncode})")
    print()
    return completed

WORKDIR = Path(tempfile.mkdtemp(prefix="budget_demo_"))
print("Working directory:", WORKDIR)


Working directory: /tmp/budget_demo_kgcowpij


### 5.1 Add, list, and report (normal case)


In [5]:
run_cli("add", "--date", "2026-09-04", "--kind", "expense",
        "--category", "food", "--amount", "18.50", "--currency", "EUR",
        workdir=WORKDIR)
run_cli("add", "--date", "2026-09-05", "--kind", "income",
        "--category", "salary", "--amount", "2500", "--currency", "EUR",
        workdir=WORKDIR)
run_cli("list", workdir=WORKDIR)
run_cli("report", workdir=WORKDIR)


$ python -m budget_manager --file transactions.json add --date 2026-09-04 --kind expense --category food --amount 18.50 --currency EUR
Added T001.
(exit 0)

$ python -m budget_manager --file transactions.json add --date 2026-09-05 --kind income --category salary --amount 2500 --currency EUR
Added T002.
(exit 0)



$ python -m budget_manager --file transactions.json list
T001  2026-09-04  expense  food                18.50 EUR
T002  2026-09-05  income   salary            2500.00 EUR
(exit 0)

$ python -m budget_manager --file transactions.json report
Transactions: 2

Currency: EUR
  Income:  2500.00
  Expense: 18.50
  Balance: 2481.50
  By category:
    salary          +2500.00
    food            -18.50
(exit 0)



CompletedProcess(args=['/workspace/aiml/.venv/bin/python3', '-m', 'budget_manager', '--file', 'transactions.json', 'report'], returncode=0, stdout='Transactions: 2\n\nCurrency: EUR\n  Income:  2500.00\n  Expense: 18.50\n  Balance: 2481.50\n  By category:\n    salary          +2500.00\n    food            -18.50\n', stderr='')

### 5.2 Rejected input (boundary)

A negative amount must fail with a specific message and leave the stored list unchanged.


In [6]:
failed = run_cli("add", "--date", "2026-09-06", "--kind", "expense",
                 "--category", "food", "--amount", "-5", "--currency", "EUR",
                 workdir=WORKDIR)
assert failed.returncode == 1
assert "amount must be > 0." in failed.stderr
run_cli("list", workdir=WORKDIR)


$ python -m budget_manager --file transactions.json add --date 2026-09-06 --kind expense --category food --amount -5 --currency EUR
Error: amount must be > 0.
(exit 1)



$ python -m budget_manager --file transactions.json list
T001  2026-09-04  expense  food                18.50 EUR
T002  2026-09-05  income   salary            2500.00 EUR
(exit 0)



CompletedProcess(args=['/workspace/aiml/.venv/bin/python3', '-m', 'budget_manager', '--file', 'transactions.json', 'list'], returncode=0, stdout='T001  2026-09-04  expense  food                18.50 EUR\nT002  2026-09-05  income   salary            2500.00 EUR\n', stderr='')

### 5.3 Import the sample CSV

Valid rows are kept. Invalid rows are reported, including the required example `Row 4: amount must be > 0.`


In [7]:
sample = PROJECT / "data" / "sample_transactions.csv"
imported = run_cli("import", str(sample), workdir=WORKDIR, filename="demo.json")
assert imported.returncode == 1
assert "Imported 5 transaction(s)." in imported.stdout
assert "Row 4: amount must be > 0." in imported.stderr
run_cli("report", workdir=WORKDIR, filename="demo.json")


$ python -m budget_manager --file demo.json import data/sample_transactions.csv
Imported 5 transaction(s).
Row 4: amount must be > 0.
Row 5: category must be one of: education, entertainment, food, freelance, gift, health, rent, salary, shopping, transport, utilities.
Row 6: date must be a real calendar date in ISO format YYYY-MM-DD.
Row 7: kind must be 'expense' or 'income'.
Row 8: amount must be > 0.
Row 9: currency must be a 3-letter ISO code (e.g. EUR).
(exit 1)

$ python -m budget_manager --file demo.json report
Transactions: 5

Currency: EUR
  Income:  2800.00
  Expense: 970.90
  Balance: 1829.10
  By category:
    salary          +2500.00
    rent            -950.00
    freelance       +300.00
    food            -18.50
    transport       -2.40
(exit 0)



CompletedProcess(args=['/workspace/aiml/.venv/bin/python3', '-m', 'budget_manager', '--file', 'demo.json', 'report'], returncode=0, stdout='Transactions: 5\n\nCurrency: EUR\n  Income:  2800.00\n  Expense: 970.90\n  Balance: 1829.10\n  By category:\n    salary          +2500.00\n    rent            -950.00\n    freelance       +300.00\n    food            -18.50\n    transport       -2.40\n', stderr='')

### 5.4 Undo after a reload (stretch goal)

`save` writes a copy, `load` reads it back into a fresh file (simulating a restart), and `undo` removes the last transaction **on disk**.


In [8]:
run_cli("save", "backup.json", workdir=WORKDIR, filename="demo.json")
refused = run_cli("save", "backup.json", workdir=WORKDIR, filename="demo.json")
assert refused.returncode == 1
assert "already exists" in refused.stderr
run_cli("save", "backup.json", "--force", workdir=WORKDIR, filename="demo.json")
run_cli("load", "backup.json", workdir=WORKDIR, filename="after_reload.json")
run_cli("list", workdir=WORKDIR, filename="after_reload.json")
run_cli("undo", workdir=WORKDIR, filename="after_reload.json")
run_cli("list", workdir=WORKDIR, filename="after_reload.json")
run_cli("report", workdir=WORKDIR, filename="after_reload.json")


$ python -m budget_manager --file demo.json save backup.json
Saved 5 transaction(s) to backup.json.
(exit 0)



$ python -m budget_manager --file demo.json save backup.json
Error: Cannot save to backup.json: file already exists. Choose a new name or use --force to overwrite it.
(exit 1)

$ python -m budget_manager --file demo.json save backup.json --force
Saved 5 transaction(s) to backup.json.
(exit 0)



$ python -m budget_manager --file after_reload.json load backup.json
Loaded 5 transaction(s) from backup.json.
(exit 0)



$ python -m budget_manager --file after_reload.json list
T001  2026-09-04  expense  food                18.50 EUR
T002  2026-09-05  income   salary            2500.00 EUR
T009  2026-09-10  income   freelance          300.00 EUR
T010  2026-09-11  expense  transport            2.40 EUR
T011  2026-09-12  expense  rent               950.00 EUR
(exit 0)



$ python -m budget_manager --file after_reload.json undo
Undone T011.
(exit 0)

$ python -m budget_manager --file after_reload.json list
T001  2026-09-04  expense  food                18.50 EUR
T002  2026-09-05  income   salary            2500.00 EUR
T009  2026-09-10  income   freelance          300.00 EUR
T010  2026-09-11  expense  transport            2.40 EUR
(exit 0)

$ python -m budget_manager --file after_reload.json report
Transactions: 4

Currency: EUR
  Income:  2800.00
  Expense: 20.90
  Balance: 2779.10
  By category:
    salary          +2500.00
    freelance       +300.00
    food            -18.50
    transport       -2.40
(exit 0)



CompletedProcess(args=['/workspace/aiml/.venv/bin/python3', '-m', 'budget_manager', '--file', 'after_reload.json', 'report'], returncode=0, stdout='Transactions: 4\n\nCurrency: EUR\n  Income:  2800.00\n  Expense: 20.90\n  Balance: 2779.10\n  By category:\n    salary          +2500.00\n    freelance       +300.00\n    food            -18.50\n    transport       -2.40\n', stderr='')

### 5.5 Interactive mode

The syllabus asks for a loop so you can enter more than one transaction without restarting. `python -m budget_manager interactive` opens a menu. The cell below simulates typing: a valid expense, a rejected negative amount, an unknown option, a report, then `q`. Exactly one transaction should be saved.


In [9]:
from budget_manager.cli import run_interactive
from budget_manager import storage

replies = iter([
    "1",
    "2026-09-04", "expense", "food", "18.50", "EUR",
    "1",
    "2026-09-06", "expense", "food", "-5", "EUR",
    "5",
    "3",
    "q",
])

def input_func(prompt=""):
    try:
        value = next(replies)
    except StopIteration:
        raise EOFError
    print(f"{prompt}{value}")
    return value

interactive_file = WORKDIR / "interactive.json"
status = run_interactive(interactive_file, input_func=input_func, output_func=print)
print(f"(exit {status})")

saved = storage.load_json(interactive_file)
print(f"Saved transactions: {len(saved)}")
assert status == 0
assert len(saved) == 1
assert saved[0].id == "T001"
assert saved[0].amount == 18.50



1) Add transaction
2) List transactions
3) Report
4) Undo last
q) Quit
Choose an option: 1
Date (YYYY-MM-DD): 2026-09-04
Kind (expense or income): expense
Category (education, entertainment, food, freelance, gift, health, rent, salary, shopping, transport, utilities): food
Amount: 18.50
Currency (e.g. EUR): EUR
Added T001.

1) Add transaction
2) List transactions
3) Report
4) Undo last
q) Quit
Choose an option: 1
Date (YYYY-MM-DD): 2026-09-06
Kind (expense or income): expense
Category (education, entertainment, food, freelance, gift, health, rent, salary, shopping, transport, utilities): food
Amount: -5
Currency (e.g. EUR): EUR
Error: amount must be > 0.

1) Add transaction
2) List transactions
3) Report
4) Undo last
q) Quit
Choose an option: 5
Unknown option, choose 1-4 or q.

1) Add transaction
2) List transactions
3) Report
4) Undo last
q) Quit
Choose an option: 3
Transactions: 1

Currency: EUR
  Income:  0.00
  Expense: 18.50
  Balance: -18.50
  By category:
    food            -1

## 6. Design notes

- **Layers:** CLI prints and parses; `services.py` has the budget rules; `validation.py` holds every field check; `storage.py` only reads and writes files.
- **Validation policy:** ambiguous values are rejected, never rewritten. `eur` stays invalid. Unknown categories are rejected with the allowed list, not stored as "other".
- **Persistence:** each change is saved with a temp file plus `os.replace`, so a crash cannot leave a half-written JSON file as the only copy. Undo is part of that same path.
- **Currencies:** totals are grouped by currency and never converted.

See `DESIGN.md` for the design rationale and the exit-ticket answers, and `AI_DISCLOSURE.md` for how an AI assistant (Grok Bot, "Mochi") was used to produce this project.
